# Experiment Results Analysis

Visualize and analyze the results from autoresearch experiment runs.
This notebook can run on the RunPod pod or locally (adjust `RESULTS_PATH` accordingly).

In [ ]:
# Load results
import pandas as pd
import matplotlib.pyplot as plt
import os

# Adjust this path to your results file
RESULTS_PATH = "/workspace/autoresearch-unified/results/climbmix/results.tsv"
# For local use: RESULTS_PATH = "results/climbmix/results.tsv"

df = pd.read_csv(RESULTS_PATH, sep="\t")
df["exp_num"] = range(len(df))
print(f"Loaded {len(df)} experiments from {RESULTS_PATH}")
df.head()

In [ ]:
# val_bpb convergence plot
fig, ax = plt.subplots(figsize=(12, 5))
colors = {"keep": "#2ecc71", "baseline": "#3498db", "discard": "#e74c3c", "crash": "#95a5a6"}
for status, group in df.groupby("status"):
    ax.scatter(group["exp_num"], group["val_bpb"], c=colors.get(status, "#999"),
               label=status, s=50, alpha=0.8, edgecolors="white", linewidth=0.5)

# Best-so-far line
kept = df[df["status"].isin(["keep", "baseline"])].copy()
kept["best_so_far"] = kept["val_bpb"].cummin()
ax.step(kept["exp_num"], kept["best_so_far"], where="post", color="#2c3e50",
        linewidth=2, label="best so far", linestyle="--")

ax.set_xlabel("Experiment #")
ax.set_ylabel("val_bpb (lower is better)")
ax.set_title("Validation BPB Convergence")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Memory vs performance
fig, ax = plt.subplots(figsize=(10, 6))
active = df[df["status"].isin(["keep", "baseline", "discard"])]
scatter = ax.scatter(active["peak_mem_gb"], active["val_bpb"],
                     c=active["tok_sec"], cmap="viridis", s=60, alpha=0.8,
                     edgecolors="white", linewidth=0.5)
plt.colorbar(scatter, label="tok/sec")
ax.set_xlabel("Peak Memory (GB)")
ax.set_ylabel("val_bpb (lower is better)")
ax.set_title("Memory Usage vs Quality")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Throughput and MFU trends
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
active = df[df["status"] != "crash"]

ax1.bar(active["exp_num"], active["tok_sec"], color="#3498db", alpha=0.7)
ax1.set_xlabel("Experiment #")
ax1.set_ylabel("Tokens/sec")
ax1.set_title("Training Throughput")
ax1.grid(True, alpha=0.3, axis="y")

ax2.bar(active["exp_num"], active["mfu"], color="#e67e22", alpha=0.7)
ax2.set_xlabel("Experiment #")
ax2.set_ylabel("MFU (%)")
ax2.set_title("Model FLOPS Utilization")
ax2.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# Cross-dataset comparison (if multiple result files exist)
import glob

results_dir = "/workspace/autoresearch-unified/results"
# For local: results_dir = "results"
all_files = glob.glob(f"{results_dir}/*/results.tsv")

if len(all_files) > 1:
    fig, ax = plt.subplots(figsize=(12, 6))
    for f in all_files:
        label = os.path.basename(os.path.dirname(f))
        d = pd.read_csv(f, sep="\t")
        kept = d[d["status"].isin(["keep", "baseline"])].copy()
        if len(kept) > 0:
            kept["best"] = kept["val_bpb"].cummin()
            ax.plot(range(len(kept)), kept["best"], label=label, linewidth=2)
    ax.set_xlabel("Kept Experiment #")
    ax.set_ylabel("Best val_bpb")
    ax.set_title("Cross-Dataset Comparison")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f"Only {len(all_files)} result file(s) found. Run on multiple datasets for comparison.")

## Export

To copy results to your local machine:
- Download `results.tsv` from the Jupyter file browser (navigate to `results/<dataset>/results.tsv`)
- Or use `scp` from your local terminal:
  ```
  scp -P <port> root@<pod-ip>:/workspace/autoresearch-unified/results/climbmix/results.tsv ./results.tsv
  ```